# 🌸 Iris Flower Classification using Machine Learning

Complete ML workflow: EDA → preprocessing → train/test split → model training → evaluation → comparison.

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")
%matplotlib inline

## 2. Load Dataset

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)
df.head()

## 3. Data Exploration

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["species"].value_counts()

In [ ]:
df.isnull().sum()

## 4. Data Visualization

### Pair Plot

In [ ]:
sns.pairplot(df, hue="species", diag_kind="hist", palette="Set2")
plt.show()

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(df.drop(columns=["species"]).corr(), annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

### Histograms

In [ ]:
df.drop(columns=["species"]).hist(figsize=(8,6), bins=15, color="#4C72B0", edgecolor="black")
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

### Box Plots

In [ ]:
melted = df.melt(id_vars="species", var_name="feature", value_name="value")
plt.figure(figsize=(9,6))
sns.boxplot(data=melted, x="feature", y="value", hue="species", palette="Set2")
plt.title("Feature Distribution by Species")
plt.xticks(rotation=20)
plt.show()

## 5. Data Preprocessing

In [ ]:
X = df.drop(columns=["species"])
y = df["species"]
X.head()

## 6. Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape, " Test size:", X_test.shape)

## 7. Model Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
print("All models trained.")

## 8. Model Evaluation

In [ ]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {
        "accuracy": acc,
        "report": classification_report(y_test, y_pred),
        "cm": confusion_matrix(y_test, y_pred, labels=model.classes_),
    }
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(results[name]["report"])

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(res["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=model.classes_, yticklabels=model.classes_, ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 9. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [r["accuracy"] for r in results.values()]
}).sort_values("Accuracy", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best model: {best_model_name} (Accuracy: {comparison_df.iloc[0]['Accuracy']:.4f})")

import joblib
joblib.dump(best_model, "iris_model.pkl")
print("Saved best model to iris_model.pkl")